<a href="https://colab.research.google.com/github/Vadapao26/WiseWaste-Data-Standardization-Engine/blob/main/Inward_ZOHO_Import.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import re
from google.colab import files
import os

print("📂 Please upload your Inward file (.xlsx or .csv):")
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
print(f"✅ Successfully uploaded: {uploaded_filename}")

# 1. Load data safely (handles CSV or Excel with any sheet name)
if uploaded_filename.endswith('.csv'):
    df_raw = pd.read_csv(uploaded_filename)
else:
    xls = pd.ExcelFile(uploaded_filename)
    print(f"📑 Available sheets in workbook: {xls.sheet_names}")
    df_raw = pd.read_excel(uploaded_filename, sheet_name=xls.sheet_names[0])

# 2. Define parent columns that need forward-filling for multi-material sub-rows
parent_cols = [
    'S.no', 'Inward Code', 'Date', 'Received material from', 'Vendor Location',
    'Source', 'Project name', 'Vehicle Number', 'Loading Cost', 'Driver Name',
    'Vehicle Vendor Name', 'Transportation Cost (in Rs)', 'Additional Cost (in Rs)', 'Total Tax (in %)'
]

existing_parent_cols = [col for col in parent_cols if col in df_raw.columns]
df_raw[existing_parent_cols] = df_raw[existing_parent_cols].ffill()

# 3. Helper function to clean filenames and strip Excel formula wrappers
def extract_clean_filename(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    match = re.search(r'HYPERLINK\("([^"]+)"', val_str, re.IGNORECASE)
    if match:
        val_str = match.group(1)
    if '/' in val_str:
        val_str = val_str.split('/')[-1]
    return val_str if val_str and val_str.lower() != 'nan' else None

# 4. Define Inward CDN Mappings & Generate Import Columns with Clean URLs
cdn_base = "https://cdn.wisewaste.in/misblobprod/1010/"

inward_photo_mapping = {
    'Vendor Invoice': ('Vendor Invoice (import)', 'vendor_invoice/'),
    'Weighment Slip': ('Weighment Slip (import)', 'weighment_slip_file/'),
    'Vehicle GPS Photo': ('Vehicle GPS Photo (Import)', 'vehicle_gps_photo/'),
    'Delivery Challan': ('Delivery Challan (import)', 'inward_record/'),
    'Material GPS Photo': ('Material GPS Photo (import)', 'material_gps/')
}

for src_col, (target_col, folder) in inward_photo_mapping.items():
    if src_col in df_raw.columns and 'Inward Code' in df_raw.columns:
        cleaned_files = df_raw[src_col].apply(extract_clean_filename)
        df_raw[target_col] = np.where(
            cleaned_files.notna(),
            cdn_base + folder + df_raw['Inward Code'].astype(str) + '/' + cleaned_files,
            None
        )

# 5. Save to Excel using openpyxl
output_file = "Inward_Zoho_Ready.xlsx"
df_raw.to_excel(output_file, sheet_name="Import Data", index=False)

# 6. Post-process with openpyxl to inject standard calculation formulas
wb = openpyxl.load_workbook(output_file)
ws = wb["Import Data"]

headers = [cell.value for cell in ws[1]]
col_map = {name: idx + 1 for idx, name in enumerate(headers) if name}

for r in range(2, ws.max_row + 1):
    # P = Received Qty, S = Rejected Qty, U = Accepted Billed Qty, Q = Rate, R = Amount
    if 'Accepted Billed Qty (in KG)' in col_map:
        ws.cell(row=r, column=col_map['Accepted Billed Qty (in KG)']).value = f"=P{r}-S{r}"
    if 'Value of Accepted Qty (in Rs)' in col_map:
        ws.cell(row=r, column=col_map['Value of Accepted Qty (in Rs)']).value = f"=U{r}*Q{r}"
    if 'Total Received Qty (in KG)' in col_map:
        ws.cell(row=r, column=col_map['Total Received Qty (in KG)']).value = f"=P{r}"
    if 'Total Accepted Qty (in KG)' in col_map:
        ws.cell(row=r, column=col_map['Total Accepted Qty (in KG)']).value = f"=P{r}-S{r}"
    if 'Total Rejected Qty (in KG)' in col_map:
        ws.cell(row=r, column=col_map['Total Rejected Qty (in KG)']).value = f"=S{r}"
    if 'Total Amount (in Rs)' in col_map:
        ws.cell(row=r, column=col_map['Total Amount (in Rs)']).value = f"=R{r}"

wb.save(output_file)
print("🎉 Processing complete! Inward data cleaned, formulas injected & plain URLs ready for Zoho.")

# 7. Auto-download the processed file
print("⬇️ Triggering automatic download...")
files.download(output_file)

📂 Please upload your Inward file (.xlsx or .csv):
